# Knowledge-conflict pilot: five methods + a BOW baseline

Runs the same five steering methods and their detection twins as
`conflict_bench/experiments/run_pilot.py` — this notebook calls the *same
functions*, so the two cannot drift apart. Use the script for a full run and
this for poking at intermediates.

| | steering | detection twin |
|---|---|---|
| prompting | `prompt_instruct` | `p_true` |
| CAD | `cad` | `context_kl` |
| AdaCAD | `adacad` | `context_jsd` |
| CK-PLUG | `ckplug` | `confidence_gain` |
| CAA | `caa` (+ `act_add`, orthogonalized) | `diffmean_proj` |
| **BOW baseline** | — | `bow` |

Plus `margin` (the DV floor) and `logit_lens` (the readout control).

Nothing is checkpointed: each cell recomputes. The margin table and the
activation cache are on disk, so re-running reuses those.

## 1. Parameters

Everything you would normally want to change lives in this one cell.

In [ ]:
CONFIG    = "../configs/pilot_five.yaml"   # relative to this notebook
OUT_DIR   = None      # None -> whatever the config says
MODEL     = None      # e.g. "google/gemma-2-2b-it" to override
LIMIT     = 200       # items, sampled evenly across relations
RELATIONS = None      # e.g. ["P17", "P27", "P36"] for a fast pass
DEVICE    = None      # None -> auto (cuda+bf16 where available, else cpu+fp32)

In [ ]:
import json, sys
from pathlib import Path

import numpy as np, pandas as pd, yaml

# make the package importable when the notebook is run from experiments/
ROOT = Path.cwd().parent.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from conflict_bench import metrics, run as runner
from conflict_bench.core import prompts
from conflict_bench.core.artifacts import ArtifactStore
from conflict_bench.core.model import ModelWrapper
from conflict_bench.experiments import run_pilot

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

cfg = yaml.safe_load(Path(CONFIG).read_text(encoding="utf-8"))
if MODEL:     cfg["model"] = dict(cfg["model"], name=MODEL)
if DEVICE:    cfg["model"]["device"] = DEVICE
if LIMIT:     cfg["limit"] = LIMIT
if RELATIONS: cfg["dataset"].setdefault("kwargs", {})["relations"] = RELATIONS
if OUT_DIR:   cfg["out_dir"] = OUT_DIR

out = Path(cfg["out_dir"]); out.mkdir(parents=True, exist_ok=True)
artifacts = ArtifactStore(out / "artifacts")
prompts.configure(**(cfg.get("prompt") or {}))
print("out_dir:", out.resolve())
print("detectors:", cfg["detectors"])
print("steerers :", cfg["steerers"])

## 2. Model

Loading is the slow part; keep this cell separate so the kernel holds the model across edits below.

In [ ]:
model = ModelWrapper.from_config(
    cfg, activation_cache_dir=out / "activations",
    margin_table_path=out / "margin_table.jsonl")
# loading the model binds the tokenizer to core/prompts, which is what makes
# `chat: auto` resolve - so read the prompt spec only after this cell
print("prompt spec:", prompts.DEFAULT.as_dict())
model.describe()

## 3. Data and behavioural labels

The label is the model's *generated* answer under C matched against the
counterfactual aliases — not a teacher-forced score. Check the
context-following rate before going further: near 0 or 1 and every trainable
method (BOW, the probes, the CAA vectors) has no contrast to fit and will
say so.

In [ ]:
items, labels = run_pilot.load_data(cfg, model, out)
groups = [it.relation for it in items]
print(f"{len(items)} items | {len(set(groups))} relations | "
      f"context-following rate = {np.mean(labels):.3f}")
pd.read_csv(out / "labels.csv").head()

## 4. The N/S/C/R design

All four conditions, raw and R-corrected. Read two things here:

- **`context_win_rate` under R vs under N** — R is supposed to be *only* a
  format baseline. If it moves the margin much beyond what N does, R is doing
  more than formatting and the correction is suspect.
- **`sd_margin` vs `sd_margin_corrected`** — if the corrected spread is far
  smaller, the raw DV was mostly a per-item answer-string constant.

In [ ]:
cond_summary = runner.run_conditions(cfg, model, items, labels, out)
display(cond_summary)

cm = pd.read_csv(out / "condition_margins.csv")
print("corr(margin_C, margin_R) =", round(cm.margin_C.corr(cm.margin_R), 5),
      "  <- near 1.0 means the raw margin is dominated by the format constant")

## 5. Detection — including the baseline that matters

Three nulls, all reported beside each detector:

- `base_rate_auroc` — knows only the relation.
- **`bow`** — TF-IDF over the passage and question, no model internals. If a
  probe cannot beat it, the probe found a property of the *items*, not of the
  model.
- `auroc_over_readout` — how much a probe beats the logit lens at the same
  layer and position. Near zero at `last` is expected; a positive value at
  `end_of_context` is the claim worth making.

In [ ]:
det_summ = runner.run_detection(cfg, model, items, labels, groups,
                               out, artifacts)
det = pd.DataFrame(det_summ).T
# `task` says which question was asked; `condition` is "instance" under the
# conflict task, because S and C are the two classes rather than a sweep.
# auroc_raw / auroc_corrected are the two views of the R correction - `view`
# says which one `auroc` mirrors.
cols = [c for c in ["method", "task", "position", "layer", "view", "auroc",
                    "auroc_std", "base_rate_auroc", "readout_auroc",
                    "auroc_over_readout", "auroc_raw", "auroc_corrected",
                    "positive_rate", "n_instances", "is_readout_position",
                    "n_folds"] if c in det.columns]
det[cols].sort_values("auroc", ascending=False)

In [ ]:
det_table = metrics.simple_report(det_summ).set_index("method")
task = cfg.get("detection_task", "conflict")
bow_auroc = det_table.auroc.get("bow", float("nan"))

if task == "conflict":
    print("NOTE: under the conflict task BOW is NOT a null.")
    print('"Does this passage contradict?" is partly decidable from the')
    print("passage text alone, so BOW is closer to an oracle than a floor.")
    print("Read it as a ceiling on the text-only route.\n")

print(f"BOW AUROC = {bow_auroc:.3f}\n")
for name in ["margin", "p_true", "context_kl", "context_jsd",
             "confidence_gain", "diffmean_proj", "linear_probe"]:
    if name not in det_table.index: continue
    a = det_table.auroc[name]
    verdict = "above BOW" if a > bow_auroc else "below BOW"
    print(f"  {name:18s} {a:.3f}   {verdict}")

## 6. Steering

The question: **given a conflicting passage, can the intervention make the
model output the conflicting answer when it otherwise would not?**

Every method is scored on the same DV - the multi-token teacher-forced margin -
and runs under **both** conditions:

* `C` the measurement, the conflict the method is meant to resolve
* `S` the matched specificity control: same item, same dose, but a passage
  that agrees with memory. `conflict_specific_effect` is the difference.

Raw margins are primary; the R-corrected view rides alongside. Watch
`flip_rate_raw` against `flip_rate_corrected` - a large gap means the flips
come from the reference passage, not the intervention.

`cad` and `adacad` reduce exactly to the unsteered margin at strength 0.
`ckplug` does not (its neutral point is `0.5 * log p_ctx`, rank-equivalent but
not identical) - its control is the true unsteered margin regardless.

In [ ]:
steer_df = run_pilot.run_steering(cfg, model, items, labels, groups,
                                  out, artifacts)
# Always read a flip rate next to the n_flippable it was computed over: a rate
# over 6 items and a rate over 394 are not the same measurement.
cols = ["method", "target", "factor", "condition", "d_margin",
        "specific_effect", "effect_toward_target", "conflict_specific_effect",
        "flip_rate", "flip_rate_flippable", "n_flippable", "n", "has_control"]
cols = [c for c in cols if c in steer_df.columns]
steer_df[steer_df.condition == "C"][cols].sort_values(
    ["method", "target", "factor"])

## 7. Final table

The same table the script prints, written to `final_table.csv`.

In [ ]:
table = run_pilot.final_table(det_summ, steer_df, out)
table

## 8. Dose–response and the Pareto front

Two charts, never one with two y-axes:

1. **flip rate vs factor** — does turning the knob do anything?
2. **flip rate vs fluency** — the AxBench Fig-4 analogue. Up and to the right
   is better: more flips at less damage to the generation. A method that only
   climbs by moving left is buying flips with broken text.

Colours are the first five slots of the reference categorical palette in fixed
order (validated for adjacent pairs); each method also gets its own marker and
a direct end-label, so identity never rests on colour alone.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

# reference categorical palette, fixed slot order - never cycled, never ad hoc
PALETTE = {"prompt_instruct": "#2a78d6", "cad": "#eb6834", "adacad": "#1baf7a",
           "ckplug": "#eda100", "caa": "#e87ba4", "act_add": "#4a3aa7"}
MARKERS = {"prompt_instruct": "o", "cad": "s", "adacad": "^",
           "ckplug": "D", "caa": "v", "act_add": "P"}
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#d8d7d2"
TARGET = "use_context"      # flip the other way with "use_parametric"
COND = "C"                  # the conflict arm; "S" is the control arm

def style(ax, xlabel, ylabel, title):
    ax.set_title(title, color=INK, fontsize=11, pad=10, loc="left")
    ax.set_xlabel(xlabel, color=MUTED, fontsize=9)
    ax.set_ylabel(ylabel, color=MUTED, fontsize=9)
    ax.grid(True, color=GRID, linewidth=0.8, alpha=0.7)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)
    ax.tick_params(colors=MUTED, labelsize=8, length=0)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))

sub = steer_df[steer_df.target == TARGET] if len(steer_df) else steer_df
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.8), facecolor="#fcfcfb")
for ax in (ax1, ax2):
    ax.set_facecolor("#fcfcfb")

handles, ends = [], []
for method, g in sub.groupby("method"):
    g = g.sort_values("factor")
    c, mk = PALETTE.get(method, "#52514e"), MARKERS.get(method, "o")
    line, = ax1.plot(g.factor, g.flip_rate_flippable, color=c, marker=mk,
                     linewidth=2, markersize=8, label=method,
                     markeredgecolor="#fcfcfb", markeredgewidth=1.5)
    handles.append(line)
    gf = g.dropna(subset=["fluency"])
    if len(gf):
        ax2.plot(gf.fluency, gf.flip_rate_flippable, color=c, marker=mk,
                 linewidth=2, markersize=8, markeredgecolor="#fcfcfb",
                 markeredgewidth=1.5)
        last = gf.iloc[-1]
        ends.append((float(last.fluency), float(last.flip_rate_flippable),
                     method))

style(ax1, "steering factor", "flip rate (flippable items)",
      f"Dose-response - target: {TARGET}")
style(ax2, "fluency  (mean logprob/token, higher = more fluent)",
      "flip rate (flippable items)", "Pareto: flips vs fluency")

# Direct end-labels, but only for <=4 series (beyond that the legend carries
# identity on its own). They are stacked in a column at a common x to the
# right of all the data and nudged apart in y - labelling each line where it
# happens to end collides as soon as two methods land in the same place, which
# is exactly what near-identical methods like caa/act_add do.
if ends and len(ends) <= 4:
    ax2.margins(x=0.28)
    x_lab = max(x for x, _, _ in ends)
    span = max(y for _, y, _ in ends) - min(y for _, y, _ in ends)
    gap, placed = 0.06 * max(span, 0.15), []
    for x, y, method in sorted(ends, key=lambda t: t[1]):
        while any(abs(y - py) < gap for py in placed):
            y += gap
        placed.append(y)
        ax2.annotate(method, (x_lab, y), textcoords="offset points",
                     xytext=(14, 0), color=MUTED, fontsize=8, va="center",
                     annotation_clip=False)

# one figure-level legend, below the panels, so it never covers the data
fig.legend(handles=handles, loc="lower center", ncol=min(len(handles), 6),
           frameon=False, fontsize=8, labelcolor=MUTED,
           bbox_to_anchor=(0.5, -0.04))
fig.tight_layout()
plt.show()

### If the Pareto panel is empty

`fluency` is only measured for methods that generate text — `prompt_instruct`,
`caa`, `act_add`. The decoding family scores a teacher-forced margin without
generating, so they have no fluency value and appear only in the left panel.
To add them, implement CAD-family generation through a logits processor.

## 9. What to look at in the artifacts

```
out_dir/
  condition_margins.csv     N/S/C/R per item, raw and corrected
  detection_summary.csv     AUROC per (method, position) + the three nulls,
                            with auroc_raw / auroc_corrected side by side
  steering_summary.csv      per (method, target, factor, condition):
                            flip rates, n_flippable, delta-margin,
                            specific_effect, conflict_specific_effect
  final_table.csv           the final table - one row per detector, and one
                            per (steerer, target)
  artifacts/bow/*_top_features.json     <- read these
  artifacts/caa/fold*_vectors.pt        <- and these
```

`bow/top_features.json` is the quickest sanity check in the whole run. Under
the **arbitration** task, if the top features read as relation or entity-type
markers ("capital", "born", country names), then whatever predicts
context-following is topic. Under the **conflict** task expect BOW to do well
on its own merits - the passage really does carry the answer it asserts - so
read the features for whether a probe is adding anything beyond them.

In `steering_summary.csv`, check `has_control` before comparing two
`specific_effect` values: a method that declares no control gets NaN, not a
bare delta-margin dressed up as a controlled one.

In [ ]:
feat = sorted(out.glob("artifacts/bow/fold0_top_features.json"))
if feat:
    top = json.loads(feat[0].read_text(encoding="utf-8"))
    print("features pushing toward CONTEXT-following:")
    print("  ", ", ".join(w for w, _ in top["toward_context"][:15]))
    print("features pushing toward PARAMETRIC:")
    print("  ", ", ".join(w for w, _ in top["toward_parametric"][:15]))
else:
    print("no BOW artifacts yet - run the detection stage")